# bt

> Advertise matched keys over Bluetooth LE so a PC running `nadakit-receive` can type them.


In [ ]:
#| default_exp bt


In [ ]:
#| hide
from nbdev.showdoc import *


A Mac app cannot present itself as a classic Bluetooth HID keyboard. Instead this module advertises a small GATT service; each matched sound becomes a UTF-8 notify of the same key string `keys.type_key` would type. On the PC, `nadakit.receive` connects and injects those keystrokes into the focused app.

Needs Bluetooth permission for the process that launched the server (Terminal, Cursor, …).


In [ ]:
#| export
import sys, threading, time, logging
log = logging.getLogger('nadakit.bt')

def _ensure_log():
    if log.handlers: return
    h = logging.StreamHandler(sys.stderr)
    h.setFormatter(logging.Formatter('[%(asctime)s] [nadakit-bt] %(levelname)s %(message)s', '%H:%M:%S'))
    log.addHandler(h); log.setLevel(logging.INFO); log.propagate = False


In [ ]:
#| export
SERVICE = '6e616461-6b69-7400-4b42-000000000001'  # nadakit keyboard
CHAR = '6e616461-6b69-7400-4b42-000000000002'
NAME = 'nadakit'


In [ ]:
#| export
try:
    import objc
    from Foundation import NSObject, NSData
    from CoreBluetooth import (
        CBPeripheralManager, CBMutableService, CBMutableCharacteristic, CBUUID,
        CBCharacteristicPropertyRead, CBCharacteristicPropertyNotify,
        CBAttributePermissionsReadable,
        CBAdvertisementDataLocalNameKey, CBAdvertisementDataServiceUUIDsKey,
        CBManagerStatePoweredOn, CBATTErrorSuccess)
    from libdispatch import dispatch_queue_create, DISPATCH_QUEUE_SERIAL
    _cb_ok = sys.platform == 'darwin'
except Exception:
    _cb_ok = False


In [ ]:
#| export
_lock = threading.Lock()
_mgr = _char = _svc = _dele = None
_powered = _advertising = _svc_ready = False
_subs, _error = set(), None
available = _cb_ok


In [ ]:
#| export
def status():
    "Whether this Mac can advertise, is advertising, and has a PC subscribed."
    return dict(available=available, advertising=_advertising, connected=bool(_subs), error=_error)


In [ ]:
#| export
if _cb_ok:
    CBPeripheralManagerDelegate = objc.protocolNamed('CBPeripheralManagerDelegate')

    class _Delegate(NSObject, protocols=[CBPeripheralManagerDelegate]):
        def peripheralManagerDidUpdateState_(self, mgr):
            global _powered, _error
            _powered = mgr.state() == CBManagerStatePoweredOn
            log.info('bt.state powered=%s raw=%s', _powered, mgr.state())
            if not _powered: _error = 'Bluetooth is off or unavailable.'
            elif _error == 'Bluetooth is off or unavailable.': _error = None

        def peripheralManager_didAddService_error_(self, mgr, service, err):
            global _error, _svc_ready
            if err: _error = str(err)
            else: _svc_ready = True

        def peripheralManagerDidStartAdvertising_error_(self, mgr, err):
            global _advertising, _error
            if err: _advertising, _error = False, str(err); log.error('bt.advertise failed %s', err)
            else: _advertising, _error = True, None; log.info('bt.advertise ok')

        def peripheralManager_central_didSubscribeToCharacteristic_(self, mgr, central, char):
            cid = central.identifier().UUIDString(); _subs.add(cid); log.info('bt.subscribe %s count=%d', cid, len(_subs))

        def peripheralManager_central_didUnsubscribeFromCharacteristic_(self, mgr, central, char):
            cid = central.identifier().UUIDString(); _subs.discard(cid); log.info('bt.unsubscribe %s count=%d', cid, len(_subs))

        def peripheralManager_didReceiveReadRequest_(self, mgr, req):
            req.setValue_(NSData.dataWithBytes_length_(b'', 0))
            mgr.respondToRequest_withResult_(req, CBATTErrorSuccess)
else:
    _Delegate = None


In [ ]:
#| export
def _ensure():
    "Build the peripheral manager once."
    global _mgr, _char, _svc, _dele, _error, _svc_ready
    if not available: return False
    with _lock:
        if _mgr is not None: return _powered and _svc_ready
        try:
            _dele = _Delegate.alloc().init()  # keep a Python ref so callbacks survive
            q = dispatch_queue_create(b'nadakit.bt', DISPATCH_QUEUE_SERIAL)
            _mgr = CBPeripheralManager.alloc().initWithDelegate_queue_options_(_dele, q, None)
            for _ in range(80):
                if _powered: break
                time.sleep(.05)
            if not _powered:
                _error = _error or 'Bluetooth is off or unavailable.'
                return False
            su, cu = CBUUID.UUIDWithString_(SERVICE), CBUUID.UUIDWithString_(CHAR)
            _char = CBMutableCharacteristic.alloc().initWithType_properties_value_permissions_(
                cu, CBCharacteristicPropertyRead | CBCharacteristicPropertyNotify, None, CBAttributePermissionsReadable)
            _svc = CBMutableService.alloc().initWithType_primary_(su, True)
            _svc.setCharacteristics_([_char])
            _svc_ready = False
            _mgr.addService_(_svc)
            for _ in range(40):
                if _svc_ready or _error: break
                time.sleep(.05)
            return _svc_ready
        except Exception as e:
            _error = f'{type(e).__name__}: {e}'
            return False


In [ ]:
#| export
def start():
    "Advertise the keyboard service. True when advertising has been requested."
    global _error
    _ensure_log()
    log.info('bt.start requested advertising=%s powered=%s', _advertising, _powered)
    if not _ensure():
        log.error('bt.start ensure failed error=%s', _error); return False
    with _lock:
        if _advertising: log.info('bt.start already advertising'); return True
        payload = {
            CBAdvertisementDataLocalNameKey: NAME,
            CBAdvertisementDataServiceUUIDsKey: [CBUUID.UUIDWithString_(SERVICE)],
        }
        _mgr.startAdvertising_(payload)
    for _ in range(40):
        if _advertising or _error: break
        time.sleep(.05)
    if not _advertising and not _error:
        _error = 'Bluetooth advertising did not start. Allow Bluetooth for this app, then try again.'
    log.info('bt.start done advertising=%s error=%s', _advertising, _error)
    return _advertising


In [ ]:
#| export
def stop():
    "Stop advertising and drop subscribers."
    global _advertising, _error
    log.info('bt.stop advertising=%s subs=%d', _advertising, len(_subs))
    with _lock:
        if _mgr is not None and _advertising:
            _mgr.stopAdvertising()
        _advertising = False
        _subs.clear()
        if _error and 'advertising did not start' in _error: _error = None
    return True


In [ ]:
#| export
def send(k):
    "Notify connected receivers with the key string. False if nobody is subscribed."
    if not _mgr or not _char or not _subs:
        log.warning('bt.send dropped %r mgr=%s char=%s subs=%d', k[:80], bool(_mgr), bool(_char), len(_subs))
        return False
    raw = k.encode('utf-8')
    data = NSData.dataWithBytes_length_(raw, len(raw))
    _char.setValue_(data)
    ok = bool(_mgr.updateValue_forCharacteristic_onSubscribedCentrals_(data, _char, None))
    log.info('bt.send %s bytes=%d text=%r subs=%d', 'ok' if ok else 'BUSY', len(raw), k[:80], len(_subs))
    return ok
